In [84]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score
# To save models
import math
import json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint
import joblib
from contextlib import contextmanager
from scipy.stats import randint, uniform
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

In [85]:
df_prueba = "anthonny"
if df_prueba == "anthonny":
    df = pd.read_csv("../data/processed/df")
    df = df.sort_values("num_semana").reset_index(drop=True)

    weeks = df["num_semana"].unique()
    cut_w = int(len(weeks) * 0.8)

    train_weeks = weeks[:cut_w]
    test_weeks  = weeks[cut_w:]

    train = df[df["num_semana"].isin(train_weeks)]
    test  = df[df["num_semana"].isin(test_weeks)]

    X_train, y_train = train.drop(columns=["y"]), train["y"]
    X_test,  y_test  = test.drop(columns=["y"]),  test["y"]

else:
    df = pd.read_csv("../data/processed/df_ineta.cvs")
    df = df.sort_values("weekend").reset_index(drop=True) 

    X = df.drop(columns=["weekend"])
    y = df["weekend"]

    cut = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:cut], X.iloc[cut:]
    y_train, y_test = y.iloc[:cut], y.iloc[cut:]
    print ("Trabajaremos con el DF de ineta")

In [86]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0,
    allow_writing_files=False
)

cb.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

In [87]:
model = cb
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

MSE (Error cuadrático medio): 22.17
RMSE (Raíz del ECM): 4.71 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.72 % de precision


In [88]:
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

def objective(trial):
    params = {
        "iterations": 5000, # Bajamos a 2000 para las pruebas; el early stopping hará el resto
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1, 5),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": 128, # Un valor fijo suele ser suficiente para ganar tiempo
        "loss_function": "RMSE",
        "verbose": 0,
        "random_seed": 18
    }
    
    model = CatBoostRegressor(**params)
    
    model.fit(
        X_train, y_train,
        cat_features=["product"],
        eval_set=(X_test, y_test),
        early_stopping_rounds=100, # Si en 100 vueltas no mejora, pasa a la siguiente prueba
        use_best_model=True
    )
    
    preds = model.predict(X_test)
    return r2_score(y_test, preds)

# Crear el estudio y ejecutar
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) # Con 30 pruebas suele ser suficiente para ver tendencia

print(f"Mejor R2: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")


[I 2026-02-09 19:51:48,745] A new study created in memory with name: no-name-202be495-5150-410d-8821-b64d9014d6ac
[I 2026-02-09 19:51:52,518] Trial 0 finished with value: 0.717924169408729 and parameters: {'learning_rate': 0.07004346801565019, 'depth': 10, 'l2_leaf_reg': 4.318181408245579, 'random_strength': 3.002203053375071, 'bagging_temperature': 0.1883153650248114}. Best is trial 0 with value: 0.717924169408729.
[I 2026-02-09 19:51:55,226] Trial 1 finished with value: 0.720870878830482 and parameters: {'learning_rate': 0.01248561565847915, 'depth': 7, 'l2_leaf_reg': 9.687253618381119, 'random_strength': 3.4009974097822493, 'bagging_temperature': 0.47121559009260694}. Best is trial 1 with value: 0.720870878830482.
[I 2026-02-09 19:51:57,427] Trial 2 finished with value: 0.7203863454384185 and parameters: {'learning_rate': 0.030555876290261894, 'depth': 8, 'l2_leaf_reg': 1.1582529078199153, 'random_strength': 1.6309867481127136, 'bagging_temperature': 0.11907396450341334}. Best is tr

Mejor R2: 0.7272
Mejores parámetros: {'learning_rate': 0.042932912485228694, 'depth': 10, 'l2_leaf_reg': 8.338449992678601, 'random_strength': 4.605440179357166, 'bagging_temperature': 0.5040866147977144}


In [89]:
# 1. Definir el modelo con los parámetros encontrados por Optuna
best_params = study.best_params

model_final = CatBoostRegressor(
    iterations=5000,          # Subimos iteraciones para el entrenamiento final
    loss_function="RMSE",
    random_seed=18,
    verbose=100,              # Para ver el progreso cada 100 pasos
    **best_params,
    allow_writing_files=False         # Esto inserta automáticamente: depth, learning_rate, etc.
)

# 2. Entrenar (usamos early_stopping para no sobreajustar)
model_final.fit(
    X_train, y_train,
    cat_features=["product"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

0:	learn: 11.5899099	test: 8.7221957	best: 8.7221957 (0)	total: 23.4ms	remaining: 1m 56s
100:	learn: 5.7148393	test: 4.6144223	best: 4.6136878 (99)	total: 2.08s	remaining: 1m 40s
200:	learn: 5.1661450	test: 4.6752360	best: 4.6136878 (99)	total: 3.94s	remaining: 1m 34s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 4.613687834
bestIteration = 99

Shrink model to first 100 iterations.


In [90]:
pred_test = model_final.predict(X_test)
pred_train = model_final.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)
r2_train = r2_score(y_train, pred_train)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_train:.2f} % de variacion Train")
print(f"R² (Coef. determinación): {r2_test:.2f} % de variacion Test")

MSE (Error cuadrático medio): 21.29
RMSE (Raíz del ECM): 4.61 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.77 % de variacion Train
R² (Coef. determinación): 0.73 % de variacion Test


In [91]:
dump(model, open("../models/73_Cat_Boost_Regressor.pkl", "wb"))